In [ ]:
# =========================================================
# BUILD FLUX FEATURE MATRIX
# =========================================================

import os
import numpy as np
import pandas as pd

# =========================================================
# PATH
# =========================================================

FLUX_DIR = (
    r"D:\antibiotics\e-coil group"
    r"\continuous_bounded_moma_full-new"
)

OUTPUT_FILE = (
    r"D:\antibiotics\e-coil group"
    r"\flux_feature_matrix.csv"
)

# =========================================================
# PARAMETERS
# =========================================================

FEATURE_COLUMN = "ΔFlux"

# reaction considered changed
CHANGE_THRESHOLD = 1e-6

# keep reaction if changed in >= N samples
MIN_FREQUENCY = 5

# =========================================================
# LOAD ALL FLUX FILES
# =========================================================

all_rows = []

files = sorted([
    f for f in os.listdir(FLUX_DIR)
    if f.endswith("_FLUX.xlsx")
])

print("=" * 60)
print("Total flux files:")
print(len(files))
print("=" * 60)

for i, file in enumerate(files):

    print(f"[{i+1}/{len(files)}] {file}")

    path = os.path.join(
        FLUX_DIR,
        file
    )

    df = pd.read_excel(path)

    # -------------------------
    # drug pair name
    # -------------------------

    pair_name = (
        file
        .replace("_FLUX.xlsx", "")
        .upper()
    )

    row = {
        "drug_pair": pair_name
    }

    # -------------------------
    # reaction feature
    # -------------------------

    for _, r in df.iterrows():

        rxn = str(r["Reaction"])

        value = r[FEATURE_COLUMN]

        row[rxn] = value

    all_rows.append(row)

# =========================================================
# BUILD MATRIX
# =========================================================

X = pd.DataFrame(all_rows)

X = X.fillna(0)

print("\nRaw matrix shape:")
print(X.shape)

# =========================================================
# REMOVE ZERO-VARIANCE REACTIONS
# =========================================================

feature_cols = [
    c for c in X.columns
    if c != "drug_pair"
]

stds = X[feature_cols].std()

keep_std = stds > 0

kept_cols = stds[keep_std].index.tolist()

X = X[
    ["drug_pair"] +
    kept_cols
]

print("\nAfter removing zero-variance reactions:")
print(X.shape)

# =========================================================
# REACTION FREQUENCY FILTER
# =========================================================

feature_cols = [
    c for c in X.columns
    if c != "drug_pair"
]

freq = (
    np.abs(
        X[feature_cols]
    ) > CHANGE_THRESHOLD
).sum(axis=0)

keep_freq = freq >= MIN_FREQUENCY

kept_cols = freq[
    keep_freq
].index.tolist()

X = X[
    ["drug_pair"] +
    kept_cols
]

print("\nAfter frequency filter:")
print(X.shape)

# =========================================================
# SAVE FEATURE MATRIX
# =========================================================

X.to_csv(
    OUTPUT_FILE,
    index=False
)

print("\nSaved:")
print(OUTPUT_FILE)

# =========================================================
# REACTION STATISTICS
# =========================================================

stats = pd.DataFrame({

    "Reaction": freq.index,

    "Frequency": freq.values

})

stats = stats.sort_values(
    "Frequency",
    ascending=False
)

stats_file = (
    r"D:\antibiotics\e-coil group"
    r"\reaction_frequency.csv"
)

stats.to_csv(
    stats_file,
    index=False
)

print("\nSaved:")
print(stats_file)

print("\nTop reactions:")

print(
    stats.head(20)
)